# ChurnIQ — Step 2: Exploratory Data Analysis & Data Understanding
**Author:** ChurnIQ ML Engineering Team  
**Role:** Senior Data Scientist & ML Engineer Mentor  
**Dataset:** IBM Telco Customer Churn (`Telco-Customer-Churn.csv`)  
**Stage:** Step 2 — Data Understanding, Quality Audit & Exploratory Data Analysis (EDA)

---

### Mentoring Note for Beginners: Why EDA Comes Before Any Modeling
In applied machine learning, beginners often rush to call `model.fit()`. In production engineering, doing so without rigorous EDA is the single biggest cause of project failure. 

Exploratory Data Analysis is not just about plotting pretty graphs; it is an forensic audit:
1. **Understanding the Business Reality:** Data represents human behavior and business policies. Knowing *why* a customer churns dictates what signals a model can learn.
2. **Detecting Silent Traps:** Dirty data (e.g. whitespace disguised as numbers, unexpected categories, target leakage) will silently corrupt model weights and produce illusionary performance that collapses in production.
3. **Establishing Baseline Hypotheses:** EDA reveals non-linearities, correlations, class imbalances, and domain patterns that directly guide feature engineering, model selection, and metric choice.


## 1. Setup & Environment Configuration
We load standard data science libraries (`pandas`, `numpy`, `matplotlib`). Notice that we set deterministic plotting aesthetics and specify path relative to project structure.


In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Matplotlib styling for professional, publication-quality visuals
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.size'] = 11
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['xtick.labelsize'] = 10
plt.rcParams['ytick.labelsize'] = 10
plt.rcParams['figure.titlesize'] = 16

print("Libraries imported successfully.")


## 2. Dataset Loading & High-Level Overview
We load the untouched raw CSV from `../data/raw/Telco-Customer-Churn.csv`. We immediately inspect its shape, columns, and sample records.


In [ ]:
# Path to the raw dataset
DATA_PATH = Path("../data/raw/Telco-Customer-Churn.csv")
assert DATA_PATH.exists(), f"Dataset not found at {DATA_PATH.resolve()}"

# Load raw dataset
df = pd.read_csv(DATA_PATH)
print(f"Dataset Dimensions: {df.shape[0]:,} rows, {df.shape[1]} columns")
df.head()


## 3. Schema & Data Dictionary
Let us inspect the data types reported by Pandas. Notice the exact storage type of each column and compare it against the expected real-world data type.


In [ ]:
# Schema overview
schema_info = pd.DataFrame({
    'Data Type': df.dtypes,
    'Non-Null Count': df.notnull().sum(),
    'Null Count': df.isnull().sum(),
    'Unique Values': df.nunique()
})
schema_info


### Data Dictionary Summary
The IBM Telco dataset tracks **7,043 customers** across **21 variables**, grouped into 4 logical domains:

1. **Identifiers & Demographics:**
   - `customerID`: Unique alphanumeric customer token (e.g., `7590-VHVEG`). Non-predictive identifier.
   - `gender`: Customer gender (`Male`, `Female`).
   - `SeniorCitizen`: Binary flag (`0` = No, `1` = Yes) indicating if the customer is a senior citizen.
   - `Partner`: Whether customer has a partner (`Yes`, `No`).
   - `Dependents`: Whether customer has dependents (`Yes`, `No`).

2. **Customer Account Information:**
   - `tenure`: Number of months the customer has stayed with the company (0 to 72 months).
   - `Contract`: The contract term (`Month-to-month`, `One year`, `Two year`).
   - `PaperlessBilling`: Whether customer enrolled in paperless billing (`Yes`, `No`).
   - `PaymentMethod`: Customer's payment mechanism (`Electronic check`, `Mailed check`, `Bank transfer (automatic)`, `Credit card (automatic)`).
   - `MonthlyCharges`: Amount charged to the customer monthly (numeric float, $18.25 to $118.75).
   - `TotalCharges`: Total amount charged over the customer's lifetime. Stored as `object`/string in the raw CSV!

3. **Subscribed Services:**
   - `PhoneService`: Whether customer has phone service (`Yes`, `No`).
   - `MultipleLines`: Whether customer has multiple lines (`No phone service`, `No`, `Yes`).
   - `InternetService`: Internet provider technology (`DSL`, `Fiber optic`, `No`).
   - `OnlineSecurity`, `OnlineBackup`, `DeviceProtection`, `TechSupport`, `StreamingTV`, `StreamingMovies`: Six internet add-on services (`No`, `Yes`, `No internet service`).

4. **Target Variable:**
   - `Churn`: Whether the customer left within the last month (`Yes`, `No`).


## 4. Missing-Value & Whitespace Analysis
Standard libraries often check missingness via `df.isnull().sum()`. However, dirty production data frequently disguises missing values as empty strings, whitespace (`" "`), or sentinel values (e.g. `None`, `N/A`, `?`). Let us perform a deep inspection.


In [ ]:
# Standard null counts
print("Standard df.isnull().sum():")
print(df.isnull().sum()[df.isnull().sum() > 0])

# Explicit whitespace check across all string columns
text_cols = df.select_dtypes(include=['object', 'str']).columns
whitespace_issues = {}
for col in text_cols:
    blank_count = (df[col].str.strip() == '').sum()
    if blank_count > 0:
        whitespace_issues[col] = blank_count

print("
Blank/Whitespace Strings Detected:")
for col, cnt in whitespace_issues.items():
    print(f"  - '{col}': {cnt} rows contain whitespace or empty strings")


### Deep Dive into the 11 Blank `TotalCharges` Records
Why are there 11 blank spaces in `TotalCharges`? Let us filter and inspect those specific records.


In [ ]:
blank_tc_rows = df[df['TotalCharges'].str.strip() == '']
print(f"Total rows with blank TotalCharges: {len(blank_tc_rows)}")
blank_tc_rows[['customerID', 'tenure', 'Contract', 'MonthlyCharges', 'TotalCharges', 'Churn']]


### Critical Insight on `TotalCharges`
Look at the `tenure` column for all 11 records: **every single one has `tenure == 0`!**
- These are brand-new customers who signed up in the current billing cycle and have not yet completed their first month of service.
- Because no invoice has completed a billing cycle, their accumulated lifetime billing is logically **$0.00**.
- All 11 customers have `Churn == 'No'` (they just joined, so they haven't churned).
- 10 of them have a `Two year` contract, and 1 has a `One year` contract.

**Data Cleaning Decision:**
Rather than destructively discarding these 11 customer records (as many naive tutorials do), we impute `TotalCharges = 0.0` when `tenure == 0`. This preserves 100% of our sample size without introducing bias.

Let us prepare a working numerical copy of `TotalCharges` for our exploratory analysis:


In [ ]:
# Create numeric working copy for EDA
df['TotalCharges_num'] = pd.to_numeric(df['TotalCharges'].str.strip(), errors='coerce')
# Impute 0.0 for tenure == 0
df.loc[df['tenure'] == 0, 'TotalCharges_num'] = 0.0
# Binary target flag for convenience in correlation calculations
df['Churn_binary'] = (df['Churn'] == 'Yes').astype(int)

print("Numeric TotalCharges prepared. Remaining nulls:", df['TotalCharges_num'].isnull().sum())


## 5. Duplicate Rows & Identifier Uniqueness Checks
We verify whether any customers were entered twice or if duplicate rows exist.


In [ ]:
exact_duplicates = df.duplicated().sum()
id_duplicates = df['customerID'].duplicated().sum()

print(f"Exact Duplicate Rows across all columns: {exact_duplicates}")
print(f"Duplicate Customer IDs: {id_duplicates}")
print(f"Total Unique Customer IDs: {df['customerID'].nunique():,} / {len(df):,}")


## 6. Target Class Distribution & Overall Churn Percentage
The target variable is `Churn` (`'Yes'` vs `'No'`). Let us quantify the distribution and visualize the class balance.


In [ ]:
churn_counts = df['Churn'].value_counts()
churn_percentages = df['Churn'].value_counts(normalize=True) * 100

print("Target Class Breakdown:")
for label in ['No', 'Yes']:
    print(f"  {label}: {churn_counts[label]:,} customers ({churn_percentages[label]:.2f}%)")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Subplot 1: Bar Chart
bars = ax1.bar(churn_counts.index, churn_counts.values, color=['#2b5c8f', '#d95f02'], width=0.5, edgecolor='black')
ax1.set_title("Customer Churn Count Distribution")
ax1.set_xlabel("Churn Status")
ax1.set_ylabel("Number of Customers")
ax1.set_ylim(0, 6000)
for bar in bars:
    yval = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2.0, yval + 100, f'{yval:,}\n({yval/len(df)*100:.1f}%)', 
             ha='center', va='bottom', fontweight='bold')

# Subplot 2: Donut Chart
wedges, texts, autotexts = ax2.pie(
    churn_counts.values, 
    labels=['Retained (No)', 'Churned (Yes)'], 
    autopct='%1.2f%%', 
    colors=['#2b5c8f', '#d95f02'],
    startangle=90, 
    wedgeprops=dict(width=0.4, edgecolor='white', linewidth=2)
)
for at in autotexts:
    at.set_color('white')
    at.set_weight('bold')
ax2.set_title("Overall Churn Percentage")

plt.tight_layout()
plt.show()


### Mentoring Takeaway: Class Imbalance (~73.5% vs ~26.5%)
In this dataset, roughly **1 out of every 4 customers churns** (26.54%). 
- A naive baseline classifier that always predicts "No Churn" would achieve **73.46% accuracy** while catching exactly **zero** churning customers!
- Therefore, in later modeling stages, **Accuracy is an inappropriate evaluation metric**. We must prioritize **Recall**, **Precision**, **F1-Score**, and **PR-AUC (Precision-Recall Area Under Curve)**.


## 7. Churn by Contract Type
Contract type reflects customer commitment and switching friction. Let us examine churn rates across `Month-to-month`, `One year`, and `Two year` contracts.


In [ ]:
contract_ct = pd.crosstab(df['Contract'], df['Churn'], margins=True)
contract_rate = (df.groupby('Contract')['Churn_binary'].mean() * 100).sort_values(ascending=False)

print("Crosstab (Counts):")
print(contract_ct)
print("
Churn Rate by Contract (%):")
print(contract_rate.round(2))

# Plot
fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#d95f02' if rate > 30 else '#2b5c8f' for rate in contract_rate]
bars = ax.bar(contract_rate.index, contract_rate.values, color=colors, width=0.5, edgecolor='black')
ax.set_title("Churn Rate by Contract Term", pad=15)
ax.set_xlabel("Contract Type")
ax.set_ylabel("Churn Rate (%)")
ax.set_ylim(0, 50)
ax.axhline(26.54, color='gray', linestyle='--', linewidth=1.5, label='Overall Average Churn (26.54%)')

for bar in bars:
    y = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2.0, y + 1.2, f'{y:.2f}%', ha='center', va='bottom', fontweight='bold')

ax.legend(loc='upper right')
plt.tight_layout()
plt.show()


### Findings & Business Interpretation: Contract Type
- **Month-to-month:** Staggering **42.71% churn rate** (1,655 out of 3,875 customers churned). Month-to-month customers represent **88.5% of all churn events**!
- **One year:** Churn drops to **11.27%**.
- **Two year:** Churn plummets to **2.83%**.
- **Insight:** Long-term contracts introduce switching barriers and commitment. Month-to-month contracts lack lock-in, making these customers highly sensitive to competitive marketing, billing dissatisfaction, and service issues.


## 8. Churn by Customer Tenure
Tenure measures customer relationship duration in months. How does tenure influence customer retention?


In [ ]:
# Tenure distributions
retained_tenure = df[df['Churn'] == 'No']['tenure']
churned_tenure = df[df['Churn'] == 'Yes']['tenure']

print("Tenure Summary Statistics by Churn:")
print(f"  Retained (No):  Mean = {retained_tenure.mean():.1f} months | Median = {retained_tenure.median():.1f} months")
print(f"  Churned  (Yes): Mean = {churned_tenure.mean():.1f} months | Median = {churned_tenure.median():.1f} months")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Histogram / Distribution
ax1.hist(retained_tenure, bins=30, alpha=0.6, label='Retained (No)', color='#2b5c8f', edgecolor='black')
ax1.hist(churned_tenure, bins=30, alpha=0.7, label='Churned (Yes)', color='#d95f02', edgecolor='black')
ax1.set_title("Tenure Distribution by Churn Status")
ax1.set_xlabel("Tenure (Months)")
ax1.set_ylabel("Customer Count")
ax1.legend()

# Tenure Bins Analysis
df['tenure_bin'] = pd.cut(df['tenure'], bins=[-1, 12, 24, 48, 72], labels=['0-12 Mos', '13-24 Mos', '25-48 Mos', '49-72 Mos'])
bin_rate = df.groupby('tenure_bin', observed=False)['Churn_binary'].mean() * 100

bars = ax2.bar(bin_rate.index.astype(str), bin_rate.values, color=['#d95f02', '#e6ab02', '#7570b3', '#2b5c8f'], width=0.5, edgecolor='black')
ax2.set_title("Churn Rate by Tenure Cohort")
ax2.set_xlabel("Tenure Cohort")
ax2.set_ylabel("Churn Rate (%)")
ax2.set_ylim(0, 60)
ax2.axhline(26.54, color='gray', linestyle='--', label='Overall Average (26.54%)')

for bar in bars:
    y = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2.0, y + 1.2, f'{y:.2f}%', ha='center', va='bottom', fontweight='bold')
ax2.legend()

plt.tight_layout()
plt.show()


### Findings & Business Interpretation: Customer Tenure
- **The First-Year Cliff:** Customers in their first 12 months exhibit an alarming **47.44% churn rate**.
- **Tenure Protection:** By months 49–72, churn drops to under **9.51%**.
- **Median Contrast:** The median churned customer leaves at **10 months**, whereas retained customers have a median tenure of **38 months**.
- **Insight:** The onboarding experience, early customer support, and initial service stability during the first 90 to 180 days are critical for customer lifetime value.


## 9. Churn by Internet Service
Customers are categorized into Fiber Optic, DSL, or No Internet Service. Does high-speed fiber prevent churn?


In [ ]:
internet_ct = pd.crosstab(df['InternetService'], df['Churn'], margins=True)
internet_rate = (df.groupby('InternetService')['Churn_binary'].mean() * 100).sort_values(ascending=False)

print("Internet Service Crosstab:")
print(internet_ct)
print("
Churn Rate (%):")
print(internet_rate.round(2))

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(internet_rate.index, internet_rate.values, color=['#d95f02', '#2b5c8f', '#1b9e77'], width=0.5, edgecolor='black')
ax.set_title("Churn Rate by Internet Service Type")
ax.set_xlabel("Internet Service Technology")
ax.set_ylabel("Churn Rate (%)")
ax.set_ylim(0, 50)
ax.axhline(26.54, color='gray', linestyle='--', label='Overall Average (26.54%)')

for bar in bars:
    y = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2.0, y + 1.2, f'{y:.2f}%', ha='center', va='bottom', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()


### Findings & Business Interpretation: Internet Service
- **Fiber Optic Paradox:** Fiber optic customers churn at **41.89%** (1,297 / 3,096), compared to only **18.96%** for DSL and **7.40%** for customers without internet service.
- **Why would premium fiber churn at more than double DSL?**
  - Fiber optic plans carry much higher monthly charges (typically $70–$110 vs $30–$50).
  - High charges create heightened customer expectations. Any outage, throttling, or billing discrepancy triggers immediate defection.


## 10. Churn by Payment Method
Payment methods range from automated bank deductions to manual checks.


In [ ]:
pay_rate = (df.groupby('PaymentMethod')['Churn_binary'].mean() * 100).sort_values(ascending=False)
print("Churn Rate by Payment Method (%):")
print(pay_rate.round(2))

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#d95f02' if r > 30 else '#2b5c8f' for r in pay_rate]
bars = ax.barh(pay_rate.index, pay_rate.values, color=colors, height=0.55, edgecolor='black')
ax.set_title("Churn Rate by Payment Method")
ax.set_xlabel("Churn Rate (%)")
ax.set_xlim(0, 55)
ax.axvline(26.54, color='gray', linestyle='--', label='Overall Average (26.54%)')

for bar in bars:
    w = bar.get_width()
    ax.text(w + 1.0, bar.get_y() + bar.get_height()/2.0, f'{w:.2f}%', va='center', fontweight='bold')

ax.legend(loc='lower right')
plt.tight_layout()
plt.show()


### Findings & Business Interpretation: Payment Method
- **Electronic Check:** Has an extraordinary **45.29% churn rate**.
- **Automated Methods:** Customers enrolled in `Bank transfer (automatic)` (16.71%) and `Credit card (automatic)` (15.24%) churn at almost one-third the rate of electronic check users.
- **Insight:** Manual payment methods force customers to consciously review their bill every month, creating a recurring decision point to cancel. Automated billing creates payment inertia and reduces friction.


## 11. Churn by Senior Citizen Status
Senior citizens are coded as `1` in the `SeniorCitizen` column (comprising 16.2% of the customer base).


In [ ]:
senior_ct = pd.crosstab(df['SeniorCitizen'], df['Churn'], margins=True)
senior_rate = df.groupby('SeniorCitizen')['Churn_binary'].mean() * 100

print("Senior Citizen Crosstab:")
print(senior_ct)
print("
Senior Citizen Churn Rate (%):")
print(senior_rate.round(2))

fig, ax = plt.subplots(figsize=(6, 5))
bars = ax.bar(['Non-Senior (0)', 'Senior Citizen (1)'], senior_rate.values, color=['#2b5c8f', '#d95f02'], width=0.45, edgecolor='black')
ax.set_title("Churn Rate by Senior Citizen Status")
ax.set_ylabel("Churn Rate (%)")
ax.set_ylim(0, 50)
ax.axhline(26.54, color='gray', linestyle='--', label='Overall Average (26.54%)')

for bar in bars:
    y = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2.0, y + 1.2, f'{y:.2f}%', ha='center', va='bottom', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()


### Findings: Senior Citizens
- **Senior Citizens:** Churn at **41.68%**, compared to **23.61%** for non-seniors.
- Seniors frequently experience fixed incomes (sensitivity to price increases) and may require more patient, accessible technical support.


## 12. Churn by Monthly Charges & Total Charges
Let us compare the pricing distributions of retained vs. churning customers.


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Monthly Charges Boxplot
bplot1 = ax1.boxplot([
    df[df['Churn'] == 'No']['MonthlyCharges'],
    df[df['Churn'] == 'Yes']['MonthlyCharges']
], patch_artist=True, tick_labels=['Retained (No)', 'Churned (Yes)'], widths=0.4)

bplot1['boxes'][0].set_facecolor('#2b5c8f')
bplot1['boxes'][1].set_facecolor('#d95f02')
ax1.set_title("Monthly Charges by Churn Status")
ax1.set_ylabel("Monthly Charges ($)")

# Total Charges Boxplot
bplot2 = ax2.boxplot([
    df[df['Churn'] == 'No']['TotalCharges_num'],
    df[df['Churn'] == 'Yes']['TotalCharges_num']
], patch_artist=True, tick_labels=['Retained (No)', 'Churned (Yes)'], widths=0.4)

bplot2['boxes'][0].set_facecolor('#2b5c8f')
bplot2['boxes'][1].set_facecolor('#d95f02')
ax2.set_title("Total Charges by Churn Status")
ax2.set_ylabel("Total Lifetime Charges ($)")

plt.tight_layout()
plt.show()

# Summary statistics
summary_charges = df.groupby('Churn')[['MonthlyCharges', 'TotalCharges_num']].agg(['mean', 'median', 'std'])
summary_charges.round(2)


### The Apparent Paradox Between Monthly and Total Charges
Look closely at the numbers above:
1. **Monthly Charges:** Churned customers pay **MORE** per month (Median **$79.65** vs **$64.43** for retained customers). Higher monthly recurring cost increases financial pressure and price sensitivity.
2. **Total Charges:** Churned customers paid **LESS** over their lifetime (Median **$703.55** vs **$1,683.60** for retained customers).

**Why this is NOT a contradiction:**
$TotalCharges \approx MonthlyCharges \times tenure$. Because churned customers leave rapidly (median tenure = 10 months), they never accumulate high lifetime charges. This illustrates why data scientists must always examine interactions and confounders!


## 14. Correlations, Associations & The Causality Distinction
Let us analyze correlations across numerical features and calculate **Cramér's V** for categorical features to quantify association strength with `Churn`.


In [ ]:
# 1. Pearson Correlation for Numerical Features
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges_num', 'SeniorCitizen', 'Churn_binary']
corr_matrix = df[num_cols].corr()

fig, ax = plt.subplots(figsize=(7, 6))
cax = ax.matshow(corr_matrix, cmap='coolwarm', vmin=-1, vmax=1)
fig.colorbar(cax)

ax.set_xticks(range(len(num_cols)))
ax.set_yticks(range(len(num_cols)))
ax.set_xticklabels(num_cols, rotation=45, ha='left')
ax.set_yticklabels(num_cols)
ax.set_title("Pearson Correlation Matrix", pad=20)

for i in range(len(num_cols)):
    for j in range(len(num_cols)):
        ax.text(j, i, f"{corr_matrix.iloc[i, j]:.2f}", ha="center", va="center", color="black", fontweight='bold')

plt.tight_layout()
plt.show()


In [ ]:
# 2. Cramer's V for Categorical Features
def cramers_v(x, y):
    confusion_matrix = pd.crosstab(x, y)
    n = confusion_matrix.sum().sum()
    r, k = confusion_matrix.shape
    expected = np.outer(confusion_matrix.sum(axis=1), confusion_matrix.sum(axis=0)) / n
    chi2 = np.sum((confusion_matrix.values - expected) ** 2 / expected)
    phi2 = chi2 / n
    phi2corr = max(0, phi2 - ((k - 1) * (r - 1)) / (n - 1))
    rcorr = r - ((r - 1) ** 2) / (n - 1)
    kcorr = k - ((k - 1) ** 2) / (n - 1)
    denom = min((kcorr - 1), (rcorr - 1))
    return 0.0 if denom == 0 else np.sqrt(phi2corr / denom)

cat_cols = [
    'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService',
    'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup',
    'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies',
    'Contract', 'PaperlessBilling', 'PaymentMethod'
]

associations = {col: cramers_v(df[col], df['Churn']) for col in cat_cols}
assoc_series = pd.Series(associations).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(assoc_series.index, assoc_series.values, color='#2b5c8f', height=0.6, edgecolor='black')
ax.set_title("Categorical Feature Association with Churn (Cramér's V)", pad=15)
ax.set_xlabel("Cramér's V (0 = No Association, 1 = Perfect Association)")
ax.set_xlim(0, 0.45)

for bar in bars:
    w = bar.get_width()
    ax.text(w + 0.005, bar.get_y() + bar.get_height()/2.0, f'{w:.3f}', va='center', fontsize=9)

plt.tight_layout()
plt.show()


### Critical Concept: Association vs. Causal Relationship
As an aspiring data scientist, understanding the distinction between **Association** and **Causation** is paramount:

- **Statistical Association (Correlation / Cramér's V):**  
  Measures whether two variables move together in the observed dataset. For example, `PaymentMethod == 'Electronic check'` has a high Cramér's V (0.303) and a 45.29% churn rate.
- **Causality:**  
  Requires that manipulating variable $X$ directly causes a change in outcome $Y$, holding all else equal.
- **The Pitfall:**  
  If a manager mistakenly assumes *causality*, they might mandate: *"Ban electronic checks and force all customers to pay by credit card to cut churn in half!"*  
  In reality, customers who choose electronic checks may be more financially stressed, less tech-savvy, or more prone to shopping around. Forcing a credit card payment will not solve their underlying affordability or product satisfaction issues.

**Conclusion:** Machine learning models predict based on *associations*, not causes. When interpreting models for business executives, always present risk factors as *correlates* and recommend controlled A/B testing before instituting policy interventions.


## 15. Main Findings, Business Interpretation & Modeling Next Steps

### Summary of Key Findings from Real Data
1. **The Core High-Risk Profile:**
   - **Contract:** Month-to-month contracts (42.71% churn rate, Cramér's V = 0.410 — strongest predictor).
   - **Tenure:** First 12 months of service (47.44% churn rate, Pearson $r = -0.35$).
   - **Internet Service:** Fiber Optic (41.89% churn rate).
   - **Payment Method:** Electronic Check (45.29% churn rate).
   - **Monthly Charges:** High monthly bills (Median $79.65 for churn vs $64.43 for retained).
2. **The Core Low-Risk / Sticky Profile:**
   - Two-year contracts (2.83% churn).
   - Tenure > 48 months (9.51% churn).
   - Enrolled in sticky support add-ons (`OnlineSecurity`, `TechSupport`).
   - Automated payments (`Credit card` or `Bank transfer` ~15-16% churn).
3. **Irrelevant / Zero-Signal Features:**
   - `gender` has Cramér's V = **0.000** (Churn rate: Female 26.92% vs Male 26.16%). Gender carries zero predictive signal and should not bias model decisions.
   - `PhoneService` has Cramér's V = **0.001** (virtually identical churn).

---

### Data Leakage Safeguards
Before moving to Step 3 (Feature Engineering & Model Training):
1. **`customerID` Exclusion:** Must be dropped from model features; it is an arbitrary identifier.
2. **`Churn` Target Isolation:** Must be strictly separated into vector $y$ before any transformation.
3. **Preprocessing Pipeline Isolation:** All encoding (One-Hot / Ordinal) and scaling (StandardScaler / RobustScaler) must be fit exclusively on training folds to prevent test set data leakage.

---
*Notebook complete. Ready for Step 3: Feature Engineering, Encoding, and Model Training.*
